In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
from config import *

In [2]:
import pandas as pd
import numpy as np

class DistanceCalculator:
    def __init__(self, properties, places, 
                 prop_lat_col='x', prop_lon_col='y', 
                 place_lat_col='x', place_lon_col='y', 
                 place_type_col='tipo'):
        self.properties = properties
        self.places = places
        self.prop_lat_col = prop_lat_col
        self.prop_lon_col = prop_lon_col
        self.place_lat_col = place_lat_col
        self.place_lon_col = place_lon_col
        self.place_type_col = place_type_col

    def haversine(self, lat1, lon1, lat2, lon2):
        lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
        dlon = lon2 - lon1 
        dlat = lat2 - lat1 
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a)) 
        r = 6371  # Radio de la Tierra en km
        return c * r
    
    def count_places_within_1km(self, lat, lon):
        distances = self.haversine(
            lat, lon,
            self.places[self.place_lat_col],
            self.places[self.place_lon_col]
        )
        places_within_1km = self.places[distances <= 1]
        counts = places_within_1km[self.place_type_col].value_counts()
        return counts.to_dict()
    
    def calculate(self):
        place_counts = self.properties.apply(
            lambda row: self.count_places_within_1km(
                row[self.prop_lat_col], row[self.prop_lon_col]
            ),
            axis=1
        )
        place_counts_df = pd.DataFrame(place_counts.tolist()).fillna(0)
        result = pd.concat([self.properties, place_counts_df], axis=1)
        return result

In [3]:
import pandas as pd
melbourne = pd.read_csv(os.path.join(DATA_RAW_DIR, 'Melbourne_housing_FULL.csv'))
osm = pd.read_excel(os.path.join(DATA_EXTERNAL_OSM_DIR, 'OpenStreetMap.xlsx'))

In [4]:
calc = DistanceCalculator(
    properties=melbourne,
    places=osm,
    prop_lat_col='Lattitude', prop_lon_col='Longtitude',
    place_lat_col='Lattitude', place_lon_col='Longtitude',
    place_type_col='grupo_funcional'
)

data_distance = calc.calculate()

In [5]:
data_distance.to_csv(os.path.join(DATA_EXTERNAL_DIR, 'data_joins.csv'), index = False)